## 1. Imports and Setup

In [1]:
import os
import copy
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from pathlib import Path

AUDIOSET_DIR = Path("data/audio/AudioSet")
SAMPLE_RATE  = 16000
DURATION     = 10
N_MELS       = 64
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device:      {DEVICE}")
print(f"Sample rate: {SAMPLE_RATE}Hz")
print(f"Duration:    {DURATION}s")

final_df = pd.read_csv(AUDIOSET_DIR / "final_dataset.csv")
print(f"\nTotal samples: {len(final_df)}")
print(f"\nSamples per class:")
print(final_df.groupby('cifar_class')['cifar_label'].count())

Device:      cuda
Sample rate: 16000Hz
Duration:    10s

Total samples: 993

Samples per class:
cifar_class
airplane      100
automobile    100
bird          100
cat           100
deer          100
dog           100
frog           93
horse         100
ship          100
truck         100
Name: cifar_label, dtype: int64


## 2. Dataset Class — MelSpectrogram (Baseline LSTM)

In [2]:
class AudioSetDatasetMel(Dataset):
    def __init__(self, dataframe, sample_rate=16000, 
                 duration=10, n_mels=64):
        self.df          = dataframe.reset_index(drop=True)
        self.sample_rate = sample_rate
        self.n_samples   = sample_rate * duration
        self.n_mels      = n_mels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        y, _ = librosa.load(row['filepath'], 
                            sr=self.sample_rate, 
                            duration=float(self.n_samples/self.sample_rate))

        if len(y) < self.n_samples:
            y = np.pad(y, (0, self.n_samples - len(y)))
        else:
            y = y[:self.n_samples]

        mel    = librosa.feature.melspectrogram(y=y, sr=self.sample_rate,
                                                n_mels=self.n_mels, 
                                                fmax=8000)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-9)

        # (time_steps, n_mels) for RNN
        mel_tensor = torch.tensor(mel_db.T, dtype=torch.float32)
        label      = torch.tensor(row['cifar_label'], dtype=torch.long)
        return mel_tensor, label

## 3. Dataset Class — Raw Waveform (Wav2Vec2)

In [3]:
class AudioSetDatasetWav(Dataset):
    def __init__(self, dataframe, sample_rate=16000, duration=10):
        self.df          = dataframe.reset_index(drop=True)
        self.sample_rate = sample_rate
        self.n_samples   = sample_rate * duration

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        y, _ = librosa.load(row['filepath'], 
                            sr=self.sample_rate,
                            duration=float(self.n_samples/self.sample_rate))

        if len(y) < self.n_samples:
            y = np.pad(y, (0, self.n_samples - len(y)))
        else:
            y = y[:self.n_samples]

        y = (y - y.mean()) / (y.std() + 1e-9)

        audio_tensor = torch.tensor(y, dtype=torch.float32)
        label        = torch.tensor(row['cifar_label'], dtype=torch.long)
        return audio_tensor, label

## 4. Model Definitions

# Base Model

In [4]:
import torch.nn as nn

class VATSA_AudioEncoder(nn.Module):
    def __init__(self, input_size=64, hidden_size=128, num_layers=1, 
                 num_classes=10, embedding_dim=512, dropout=0.3):
        super().__init__()
        
        self.lstm = nn.LSTM(
            input_size  = input_size,   # n_mels
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0
        )
        
        self.dropout    = nn.Dropout(dropout)
        self.projection = nn.Linear(hidden_size, embedding_dim)
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        # x: (batch, time_steps, n_mels)
        lstm_out, (hidden, _) = self.lstm(x)
        
        # Take last hidden state
        last_hidden = hidden[-1]                    # (batch, hidden_size)
        last_hidden = self.dropout(last_hidden)
        
        embedding = self.projection(last_hidden)    # (batch, embedding_dim)
        logits    = self.classifier(embedding)      # (batch, num_classes)
        
        return logits, embedding


# Quick architecture check
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = VATSA_AudioEncoder(
    input_size   = 64,
    hidden_size  = 128,
    num_layers   = 1,
    num_classes  = 10,
    embedding_dim= 512,
    dropout      = 0.3
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(f"\nModel architecture:")
print(model)

from sklearn.model_selection import StratifiedKFold
import torch.optim as optim
import copy

EPOCHS      = 30
BATCH_SIZE  = 16
LR          = 1e-3
N_FOLDS     = 5
HIDDEN_SIZE = 128
EMBED_DIM   = 512

skf     = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
labels  = final_df['cifar_label'].values

fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(final_df, labels)):
    print(f"\n{'='*50}")
    print(f"FOLD {fold+1}/{N_FOLDS}")
    print(f"{'='*50}")
    
    train_df = final_df.iloc[train_idx]
    val_df   = final_df.iloc[val_idx]
    
    train_dataset = AudioSetDatasetMel(train_df)
    val_dataset   = AudioSetDatasetMel(val_df)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, 
                              shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, 
                              shuffle=False, num_workers=0)
    
    model = VATSA_AudioEncoder(
        input_size    = 64,
        hidden_size   = HIDDEN_SIZE,
        num_layers    = 1,
        num_classes   = 10,
        embedding_dim = EMBED_DIM,
        dropout       = 0.3
    ).to(device)
    
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    best_val_acc  = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())
    
    for epoch in range(EPOCHS):
        # Train
        model.train()
        running_loss = 0.0
        for mel, label in train_loader:
            mel, label = mel.to(device), label.to(device)
            optimizer.zero_grad()
            logits, _ = model(mel)
            loss = criterion(logits, label)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        scheduler.step()
        avg_loss = running_loss / len(train_loader)
        
        # Validate
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for mel, label in val_loader:
                mel, label = mel.to(device), label.to(device)
                logits, _ = model(mel)
                correct += (logits.argmax(1) == label).sum().item()
                total   += label.size(0)
        
        val_acc = correct / total * 100
        
        if val_acc > best_val_acc:
            best_val_acc   = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
        
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{EPOCHS} - Loss: {avg_loss:.4f} - Val Acc: {val_acc:.2f}%")
    
    print(f"  Best Val Acc — Fold {fold+1}: {best_val_acc:.2f}%")
    fold_results.append(best_val_acc)

print(f"\n{'='*50}")
print(f"K-FOLD RESULTS SUMMARY")
print(f"{'='*50}")
for i, acc in enumerate(fold_results):
    print(f"  Fold {i+1}: {acc:.2f}%")
print(f"\n  Mean Accuracy: {np.mean(fold_results):.2f}%")
print(f"  Std Deviation: {np.std(fold_results):.2f}%")

Device: cuda
Total parameters: 170,506

Model architecture:
VATSA_AudioEncoder(
  (lstm): LSTM(64, 128, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (projection): Linear(in_features=128, out_features=512, bias=True)
  (classifier): Linear(in_features=512, out_features=10, bias=True)
)

FOLD 1/5


c:\Edu\VATSA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  Epoch 10/30 - Loss: 1.7188 - Val Acc: 28.64%
  Epoch 20/30 - Loss: 1.0282 - Val Acc: 28.14%
  Epoch 30/30 - Loss: 0.6786 - Val Acc: 32.66%
  Best Val Acc — Fold 1: 33.17%

FOLD 2/5
  Epoch 10/30 - Loss: 1.7048 - Val Acc: 25.13%
  Epoch 20/30 - Loss: 1.1189 - Val Acc: 21.11%
  Epoch 30/30 - Loss: 0.7509 - Val Acc: 21.61%
  Best Val Acc — Fold 2: 26.63%

FOLD 3/5
  Epoch 10/30 - Loss: 1.7094 - Val Acc: 23.12%
  Epoch 20/30 - Loss: 0.9874 - Val Acc: 23.62%
  Epoch 30/30 - Loss: 0.6282 - Val Acc: 24.12%
  Best Val Acc — Fold 3: 26.63%

FOLD 4/5
  Epoch 10/30 - Loss: 1.6658 - Val Acc: 25.76%
  Epoch 20/30 - Loss: 1.0355 - Val Acc: 24.24%
  Epoch 30/30 - Loss: 0.7255 - Val Acc: 24.24%
  Best Val Acc — Fold 4: 27.27%

FOLD 5/5
  Epoch 10/30 - Loss: 1.7120 - Val Acc: 27.78%
  Epoch 20/30 - Loss: 1.1160 - Val Acc: 23.23%
  Epoch 30/30 - Loss: 0.7289 - Val Acc: 25.25%
  Best Val Acc — Fold 5: 27.78%

K-FOLD RESULTS SUMMARY
  Fold 1: 33.17%
  Fold 2: 26.63%
  Fold 3: 26.63%
  Fold 4: 27.27%
  F

In [5]:
CIFAR10_CLASSES_LIST = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

# Retrain on full data for final per-class evaluation
# (for research purposes — real deployment would use held-out test set)
full_dataset = AudioSetDatasetMel(final_df)
full_loader  = DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Load best fold model
model.load_state_dict(best_model_wts)
model.eval()

all_preds  = []
all_labels = []

with torch.no_grad():
    for mel, label in full_loader:
        mel, label = mel.to(device), label.to(device)
        logits, _  = model(mel)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(label.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# Overall accuracy
overall_acc = (all_preds == all_labels).mean() * 100
print(f"Overall Accuracy: {overall_acc:.2f}%")
print(f"\nPer-Class Accuracy:")
print(f"{'Class':<15} {'Correct':>8} {'Total':>8} {'Accuracy':>10}")
print("-" * 45)

for i, cls in enumerate(CIFAR10_CLASSES_LIST):
    mask    = all_labels == i
    correct = (all_preds[mask] == all_labels[mask]).sum()
    total   = mask.sum()
    acc     = correct / total * 100 if total > 0 else 0
    flag    = " ⚠️" if acc < 50 else ""
    print(f"{cls:<15} {correct:>8} {total:>8} {acc:>9.2f}%{flag}")

Overall Accuracy: 32.02%

Per-Class Accuracy:
Class            Correct    Total   Accuracy
---------------------------------------------
airplane              17      100     17.00% ⚠️
automobile            42      100     42.00% ⚠️
bird                  37      100     37.00% ⚠️
cat                   19      100     19.00% ⚠️
deer                  83      100     83.00%
dog                   18      100     18.00% ⚠️
frog                  32       93     34.41% ⚠️
horse                 18      100     18.00% ⚠️
ship                   2      100      2.00% ⚠️
truck                 50      100     50.00%


In [6]:
# Save best model
torch.save({
    'model_state' : best_model_wts,
    'embedding_dim': EMBED_DIM,
    'hidden_size' : HIDDEN_SIZE,
    'input_size'  : 64,
    'num_classes' : 10,
    'fold_results': fold_results,
}, "exp6_vatsa_audio_encoder_baseline.pth")

print("✅ VATSA Audio Encoder (Baseline) saved!")

# Print experiment summary for logging
print(f"""
╔══════════════════════════════════════════════════════╗
║         EXP-006 BASELINE RNN RESULTS SUMMARY         ║
╠══════════════════════════════════════════════════════╣
║ Model        : VATSA_AudioEncoder (Single LSTM)      ║
║ Dataset      : ESC-50 mapped to CIFAR-10 (400 total) ║
║ Validation   : 5-Fold Stratified Cross Validation    ║
║ Epochs       : {EPOCHS:<3} per fold                        ║
║ Hidden Size  : {HIDDEN_SIZE:<3}                             ║
║ Embedding Dim: {EMBED_DIM:<3}                             ║
╠══════════════════════════════════════════════════════╣
║ Fold Results :                                       ║
║   Fold 1: {fold_results[0]:>6.2f}%                              ║
║   Fold 2: {fold_results[1]:>6.2f}%                              ║
║   Fold 3: {fold_results[2]:>6.2f}%                              ║
║   Fold 4: {fold_results[3]:>6.2f}%                              ║
║   Fold 5: {fold_results[4]:>6.2f}%                              ║
╠══════════════════════════════════════════════════════╣
║ Mean Accuracy: {np.mean(fold_results):>6.2f}%                         ║
║ Std Deviation: {np.std(fold_results):>6.2f}%                         ║
╚══════════════════════════════════════════════════════╝
""")

✅ VATSA Audio Encoder (Baseline) saved!

╔══════════════════════════════════════════════════════╗
║         EXP-006 BASELINE RNN RESULTS SUMMARY         ║
╠══════════════════════════════════════════════════════╣
║ Model        : VATSA_AudioEncoder (Single LSTM)      ║
║ Dataset      : ESC-50 mapped to CIFAR-10 (400 total) ║
║ Validation   : 5-Fold Stratified Cross Validation    ║
║ Epochs       : 30  per fold                        ║
║ Hidden Size  : 128                             ║
║ Embedding Dim: 512                             ║
╠══════════════════════════════════════════════════════╣
║ Fold Results :                                       ║
║   Fold 1:  33.17%                              ║
║   Fold 2:  26.63%                              ║
║   Fold 3:  26.63%                              ║
║   Fold 4:  27.27%                              ║
║   Fold 5:  27.78%                              ║
╠══════════════════════════════════════════════════════╣
║ Mean Accuracy:  28.30%          

#  wav2vec

In [7]:
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor
import torch.nn as nn

SAMPLE_RATE = 16000

# Feature extractor — handles normalisation consistently
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    "facebook/wav2vec2-base"
)
print(f"Feature extractor loaded — sample rate: {feature_extractor.sampling_rate}")
print(f"  Normalise: {feature_extractor.do_normalize}")

Feature extractor loaded — sample rate: 16000
  Normalise: True


In [8]:
class VATSA_AudioEncoder_TL(nn.Module):
    def __init__(self, embedding_dim=512, num_classes=10, 
                 freeze_backbone=True):
        super().__init__()

        self.wav2vec2 = Wav2Vec2Model.from_pretrained(
            "facebook/wav2vec2-base"
        )

        hidden_size = self.wav2vec2.config.hidden_size  # 768 for base

        self.projection = nn.Linear(hidden_size, embedding_dim)
        self.dropout    = nn.Dropout(0.3)
        self.classifier = nn.Linear(embedding_dim, num_classes)

        if freeze_backbone:
            for param in self.wav2vec2.parameters():
                param.requires_grad = False

    def forward(self, x):
        # x: (batch, time_steps) — raw waveform
        outputs     = self.wav2vec2(x)
        
        # Mean pool across time dimension
        hidden      = outputs.last_hidden_state        # (batch, seq, 768)
        pooled      = hidden.mean(dim=1)               # (batch, 768)
        
        pooled      = self.dropout(pooled)
        embedding   = self.projection(pooled)          # (batch, 512)
        logits      = self.classifier(embedding)       # (batch, 10)

        return logits, embedding


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = VATSA_AudioEncoder_TL(
    embedding_dim  = 512,
    num_classes    = 10,
    freeze_backbone= True
).to(device)

# Count trainable vs total parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() 
                       if p.requires_grad)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {total_params - trainable_params:,}")

Device: cuda


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 14175.85it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total parameters:     94,770,570
Trainable parameters: 398,858
Frozen parameters:    94,371,712


# unfrozen wav2vec

In [9]:
from sklearn.model_selection import StratifiedKFold
import torch.optim as optim
import copy

def run_kfold(full_df, freeze_backbone, stage_name, 
              epochs=30, lr=1e-3, batch_size=16):

    skf    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    labels = full_df['cifar_label'].values

    fold_results   = []
    best_model_wts = None
    best_overall   = 0.0

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(full_df, labels)
    ):
        print(f"\n{'='*50}")
        print(f"{stage_name} — FOLD {fold+1}/5")
        print(f"{'='*50}")

        train_df = full_df.iloc[train_idx]
        val_df   = full_df.iloc[val_idx]

        train_loader = DataLoader(
            AudioSetDatasetWav(train_df), 
            batch_size=batch_size, shuffle=True,  num_workers=0
        )
        val_loader = DataLoader(
            AudioSetDatasetWav(val_df),
            batch_size=batch_size, shuffle=False, num_workers=0
        )

        model = VATSA_AudioEncoder_TL(
            embedding_dim  = 512,
            num_classes    = 10,
            freeze_backbone= freeze_backbone
        ).to(device)

        optimizer = optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr, weight_decay=1e-4
        )
        criterion = nn.CrossEntropyLoss()
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=epochs
        )

        best_val_acc   = 0.0
        best_fold_wts  = copy.deepcopy(model.state_dict())

        for epoch in range(epochs):
            model.train()
            running_loss = 0.0

            for audio, label in train_loader:
                audio, label = audio.to(device), label.to(device)
                optimizer.zero_grad()
                logits, _ = model(audio)
                loss = criterion(logits, label)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()

            scheduler.step()
            avg_loss = running_loss / len(train_loader)

            model.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for audio, label in val_loader:
                    audio, label = audio.to(device), label.to(device)
                    logits, _    = model(audio)
                    correct += (logits.argmax(1) == label).sum().item()
                    total   += label.size(0)

            val_acc = correct / total * 100

            if val_acc > best_val_acc:
                best_val_acc  = val_acc
                best_fold_wts = copy.deepcopy(model.state_dict())

            if (epoch + 1) % 10 == 0:
                print(f"  Epoch {epoch+1}/{epochs} - "
                      f"Loss: {avg_loss:.4f} - "
                      f"Val Acc: {val_acc:.2f}%")

        print(f"  Best Val Acc — Fold {fold+1}: {best_val_acc:.2f}%")
        fold_results.append(best_val_acc)

        if best_val_acc > best_overall:
            best_overall   = best_val_acc
            best_model_wts = best_fold_wts

    print(f"\n{'='*50}")
    print(f"{stage_name} — K-FOLD SUMMARY")
    print(f"{'='*50}")
    for i, acc in enumerate(fold_results):
        print(f"  Fold {i+1}: {acc:.2f}%")
    print(f"\n  Mean: {np.mean(fold_results):.2f}%")
    print(f"  Std:  {np.std(fold_results):.2f}%")

    return fold_results, best_model_wts


# Run Stage 1 — frozen backbone
stage1_results, stage1_weights = run_kfold(
    final_df,
    freeze_backbone = True,
    stage_name      = "Stage 1 — Frozen Backbone",
    epochs          = 30,
    lr              = 1e-3
)


Stage 1 — Frozen Backbone — FOLD 1/5


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 17486.97it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch 10/30 - Loss: 1.5739 - Val Acc: 28.14%
  Epoch 20/30 - Loss: 1.4320 - Val Acc: 28.64%
  Epoch 30/30 - Loss: 1.3672 - Val Acc: 27.64%
  Best Val Acc — Fold 1: 30.15%

Stage 1 — Frozen Backbone — FOLD 2/5


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 10101.68it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch 10/30 - Loss: 1.6022 - Val Acc: 24.12%
  Epoch 20/30 - Loss: 1.4151 - Val Acc: 22.61%
  Epoch 30/30 - Loss: 1.3500 - Val Acc: 21.11%
  Best Val Acc — Fold 2: 26.13%

Stage 1 — Frozen Backbone — FOLD 3/5


...

  Epoch 10/30 - Loss: 1.6160 - Val Acc: 33.67%
  Epoch 20/30 - Loss: 1.4604 - Val Acc: 33.17%
  Epoch 30/30 - Loss: 1.3775 - Val Acc: 34.67%
  Best Val Acc — Fold 3: 35.68%

Stage 1 — Frozen Backbone — FOLD 4/5


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 15182.94it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch 10/30 - Loss: 1.5965 - Val Acc: 25.25%
  Epoch 20/30 - Loss: 1.4306 - Val Acc: 27.27%
  Epoch 30/30 - Loss: 1.3459 - Val Acc: 26.26%
  Best Val Acc — Fold 4: 27.78%

Stage 1 — Frozen Backbone — FOLD 5/5


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 16440.61it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch 10/30 - Loss: 1.5960 - Val Acc: 29.29%
  Epoch 20/30 - Loss: 1.4431 - Val Acc: 29.80%
  Epoch 30/30 - Loss: 1.3700 - Val Acc: 29.29%
  Best Val Acc — Fold 5: 32.32%

Stage 1 — Frozen Backbone — K-FOLD SUMMARY
  Fold 1: 30.15%
  Fold 2: 26.13%
  Fold 3: 35.68%
  Fold 4: 27.78%
  Fold 5: 32.32%

  Mean: 30.41%
  Std:  3.37%


# partial unfreezed

In [10]:
def unfreeze_top_layers(model, num_layers=4):
    # Wav2Vec2 base has 12 transformer layers
    # Unfreeze last num_layers
    for param in model.wav2vec2.parameters():
        param.requires_grad = False

    encoder_layers = model.wav2vec2.encoder.layers
    for layer in encoder_layers[-num_layers:]:
        for param in layer.parameters():
            param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() 
                    if p.requires_grad)
    print(f"Trainable parameters after unfreeze: {trainable:,}")


# Patch run_kfold to support partial unfreeze
def run_kfold_unfreeze(full_df, num_layers, stage_name,
                        epochs=30, lr=1e-4, batch_size=16):

    skf    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    labels = full_df['cifar_label'].values

    fold_results   = []
    best_model_wts = None
    best_overall   = 0.0

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(full_df, labels)
    ):
        print(f"\n{'='*50}")
        print(f"{stage_name} — FOLD {fold+1}/5")
        print(f"{'='*50}")

        train_df = full_df.iloc[train_idx]
        val_df   = full_df.iloc[val_idx]

        train_loader = DataLoader(
            AudioSetDatasetWav(train_df),
            batch_size=batch_size, shuffle=True,  num_workers=0
        )
        val_loader = DataLoader(
            AudioSetDatasetWav(val_df),
            batch_size=batch_size, shuffle=False, num_workers=0
        )

        # Load stage 1 weights as starting point
        model = VATSA_AudioEncoder_TL(
            embedding_dim  = 512,
            num_classes    = 10,
            freeze_backbone= True
        ).to(device)
        model.load_state_dict(stage1_weights)

        # Unfreeze top layers
        unfreeze_top_layers(model, num_layers=num_layers)

        optimizer = optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr, weight_decay=1e-4  # lower lr for fine-tuning
        )
        criterion = nn.CrossEntropyLoss()
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=epochs
        )

        best_val_acc  = 0.0
        best_fold_wts = copy.deepcopy(model.state_dict())

        for epoch in range(epochs):
            model.train()
            running_loss = 0.0

            for audio, label in train_loader:
                audio, label = audio.to(device), label.to(device)
                optimizer.zero_grad()
                logits, _ = model(audio)
                loss = criterion(logits, label)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()

            scheduler.step()
            avg_loss = running_loss / len(train_loader)

            model.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for audio, label in val_loader:
                    audio, label = audio.to(device), label.to(device)
                    logits, _    = model(audio)
                    correct += (logits.argmax(1) == label).sum().item()
                    total   += label.size(0)

            val_acc = correct / total * 100

            if val_acc > best_val_acc:
                best_val_acc  = val_acc
                best_fold_wts = copy.deepcopy(model.state_dict())

            if (epoch + 1) % 10 == 0:
                print(f"  Epoch {epoch+1}/{epochs} - "
                      f"Loss: {avg_loss:.4f} - "
                      f"Val Acc: {val_acc:.2f}%")

        print(f"  Best Val Acc — Fold {fold+1}: {best_val_acc:.2f}%")
        fold_results.append(best_val_acc)

        if best_val_acc > best_overall:
            best_overall   = best_val_acc
            best_model_wts = best_fold_wts

    print(f"\n{'='*50}")
    print(f"{stage_name} — K-FOLD SUMMARY")
    print(f"{'='*50}")
    for i, acc in enumerate(fold_results):
        print(f"  Fold {i+1}: {acc:.2f}%")
    print(f"\n  Mean: {np.mean(fold_results):.2f}%")
    print(f"  Std:  {np.std(fold_results):.2f}%")

    return fold_results, best_model_wts


# Run Stage 2 — unfreeze last 4 transformer layers
stage2_results, stage2_weights = run_kfold_unfreeze(
    final_df,
    num_layers = 4,
    stage_name = "Stage 2 — Partial Unfreeze (4 layers)",
    epochs     = 30,
    lr         = 1e-4   # lower lr for fine-tuning
)


Stage 2 — Partial Unfreeze (4 layers) — FOLD 1/5


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 16450.70it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable parameters after unfreeze: 28,750,346
  Epoch 10/30 - Loss: 0.4921 - Val Acc: 25.13%
  Epoch 20/30 - Loss: 0.1109 - Val Acc: 27.64%
  Epoch 30/30 - Loss: 0.0905 - Val Acc: 29.65%
  Best Val Acc — Fold 1: 31.66%

Stage 2 — Partial Unfreeze (4 layers) — FOLD 2/5


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 13811.26it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable parameters after unfreeze: 28,750,346
  Epoch 10/30 - Loss: 0.5106 - Val Acc: 29.65%
  Epoch 20/30 - Loss: 0.1507 - Val Acc: 25.63%
  Epoch 30/30 - Loss: 0.0842 - Val Acc: 29.65%
  Best Val Acc — Fold 2: 32.16%

Stage 2 — Partial Unfreeze (4 layers) — FOLD 3/5


...

Trainable parameters after unfreeze: 28,750,346
  Epoch 10/30 - Loss: 0.5488 - Val Acc: 34.17%
  Epoch 20/30 - Loss: 0.1246 - Val Acc: 34.17%
  Epoch 30/30 - Loss: 0.0921 - Val Acc: 35.68%
  Best Val Acc — Fold 3: 37.69%

Stage 2 — Partial Unfreeze (4 layers) — FOLD 4/5


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 9075.14it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable parameters after unfreeze: 28,750,346
  Epoch 10/30 - Loss: 0.5361 - Val Acc: 32.83%
  Epoch 20/30 - Loss: 0.1119 - Val Acc: 28.79%
  Epoch 30/30 - Loss: 0.0797 - Val Acc: 32.83%
  Best Val Acc — Fold 4: 33.33%

Stage 2 — Partial Unfreeze (4 layers) — FOLD 5/5


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 14151.82it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable parameters after unfreeze: 28,750,346
  Epoch 10/30 - Loss: 0.5897 - Val Acc: 32.32%
  Epoch 20/30 - Loss: 0.1243 - Val Acc: 33.33%
  Epoch 30/30 - Loss: 0.0798 - Val Acc: 32.32%
  Best Val Acc — Fold 5: 37.88%

Stage 2 — Partial Unfreeze (4 layers) — K-FOLD SUMMARY
  Fold 1: 31.66%
  Fold 2: 32.16%
  Fold 3: 37.69%
  Fold 4: 33.33%
  Fold 5: 37.88%

  Mean: 34.54%
  Std:  2.70%


In [11]:
CIFAR10_CLASSES_LIST = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

full_dataset = AudioSetDatasetWav(final_df)
full_loader  = DataLoader(full_dataset, batch_size=16, shuffle=False)

model_eval = VATSA_AudioEncoder_TL(
    embedding_dim=512, num_classes=10, freeze_backbone=True
).to(device)
model_eval.load_state_dict(stage2_weights)
model_eval.eval()

all_preds  = []
all_labels = []

with torch.no_grad():
    for audio, label in full_loader:
        audio, label = audio.to(device), label.to(device)
        logits, _    = model_eval(audio)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(label.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

overall_acc = (all_preds == all_labels).mean() * 100
print(f"Overall Accuracy: {overall_acc:.2f}%")
print(f"\nPer-Class Accuracy:")
print(f"{'Class':<15} {'Correct':>8} {'Total':>8} {'Accuracy':>10}")
print("-" * 45)

for i, cls in enumerate(CIFAR10_CLASSES_LIST):
    mask    = all_labels == i
    correct = (all_preds[mask] == all_labels[mask]).sum()
    total   = mask.sum()
    acc     = correct / total * 100 if total > 0 else 0
    flag    = " ⚠️" if acc < 50 else ""
    print(f"{cls:<15} {correct:>8} {total:>8} {acc:>9.2f}%{flag}")

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 10149.18it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Overall Accuracy: 40.58%

Per-Class Accuracy:
Class            Correct    Total   Accuracy
---------------------------------------------
airplane              29      100     29.00% ⚠️
automobile            46      100     46.00% ⚠️
bird                  61      100     61.00%
cat                   45      100     45.00% ⚠️
deer                  60      100     60.00%
dog                   11      100     11.00% ⚠️
frog                  52       93     55.91%
horse                 40      100     40.00% ⚠️
ship                  44      100     44.00% ⚠️
truck                 15      100     15.00% ⚠️


In [12]:
print("=" * 55)
print("VATSA AUDIO MODULE — BENCHMARK COMPARISON")
print("=" * 55)
print(f"{'Experiment':<35} {'Mean Acc':>10} {'Std':>8}")
print("-" * 55)
print(f"{'Baseline LSTM (scratch)':<35} "
      f"{np.mean([50.00,58.75,53.75,55.00,46.25]):>9.2f}% "
      f"{np.std([50.00,58.75,53.75,55.00,46.25]):>7.2f}%")
print(f"{'Wav2Vec2 Frozen (Stage 1)':<35} "
      f"{np.mean(stage1_results):>9.2f}% "
      f"{np.std(stage1_results):>7.2f}%")
print(f"{'Wav2Vec2 Partial Unfreeze (Stage 2)':<35} "
      f"{np.mean(stage2_results):>9.2f}% "
      f"{np.std(stage2_results):>7.2f}%")
print("=" * 55)
print(f"\nDelta (Baseline → Stage 2): "
      f"+{np.mean(stage2_results) - 52.75:.2f}%")

VATSA AUDIO MODULE — BENCHMARK COMPARISON
Experiment                            Mean Acc      Std
-------------------------------------------------------
Baseline LSTM (scratch)                 52.75%    4.29%
Wav2Vec2 Frozen (Stage 1)               30.41%    3.37%
Wav2Vec2 Partial Unfreeze (Stage 2)     34.54%    2.70%

Delta (Baseline → Stage 2): +-18.21%


In [13]:
torch.save({
    'model_state'  : stage2_weights,
    'embedding_dim': 512,
    'hidden_size'  : 768,
    'input_size'   : 16000 * 5,
    'num_classes'  : 10,
    'stage1_results': stage1_results,
    'stage2_results': stage2_results,
}, "exp6_vatsa_audio_encoder_transfer.pth")

print("✅ VATSA Audio Encoder (Transfer Learning) saved!")

✅ VATSA Audio Encoder (Transfer Learning) saved!
